In [19]:
# ============================================================
# IMPORTAÇÕES
# ============================================================

import re
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def sanitizar_nome(nome):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(nome))


def formatar_count(valor):
    return f"{int(valor):,}".replace(",", ".")


def fig_plotly_to_html(fig):
    return pio.to_html(
        fig,
        full_html=False,
        include_plotlyjs=False,
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "responsive": True,
            "displaylogo": False,
            "modeBarButtonsToRemove": ["lasso2d", "select2d"]
        }
    )


def gerar_elipsoide_3d_media_cov(media, cov, raio=3.0, n_u=72, n_v=36):
    """
    Gera uma elipsoide 3D baseada em média e matriz de covariância.

    A superfície representa:

        (x - mu)' Sigma^{-1} (x - mu) = raio^2

    raio:
        1.0 -> região central
        2.0 -> região moderada
        3.0 -> região ampla
    """

    autovalores, autovetores = np.linalg.eigh(cov)

    autovalores = np.maximum(autovalores, 1e-12)

    u = np.linspace(0, 2 * np.pi, n_u)
    v = np.linspace(0, np.pi, n_v)

    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))

    esfera = np.stack([x, y, z], axis=-1)

    transformacao = autovetores @ np.diag(np.sqrt(autovalores) * raio)

    elipsoide = esfera @ transformacao.T + media

    return (
        elipsoide[:, :, 0],
        elipsoide[:, :, 1],
        elipsoide[:, :, 2]
    )


# ============================================================
# FUNÇÃO PRINCIPAL
# ============================================================

def gerar_relatorio_3x3_interativo(
    rank,
    arquivo_dados="creditcard.csv",
    arquivo_scores="3x3_scores.csv",
    pasta_saida=".",
    nome_base_saida="3x3_relat_interativo",
    raio_elipsoide=3.0
):

    # ========================================================
    # LEITURA DOS DADOS
    # ========================================================

    df = pd.read_csv(arquivo_dados)
    scores_3x3 = pd.read_csv(arquivo_scores)

    if "status_fraude" in df.columns:
        target_name = "status_fraude"

    elif "Class" in df.columns:
        df = df.rename(columns={"Class": "status_fraude"})
        target_name = "status_fraude"

    else:
        raise ValueError("Não encontrei a coluna target: 'status_fraude' ou 'Class'.")

    # ========================================================
    # VALIDAÇÃO DO CSV DE SCORES
    # ========================================================

    colunas_necessarias = [
        "Feature_1",
        "Feature_2",
        "Feature_3",
        "Posicao_Rank",
        "AUC_PR",
        "MCC",
        "Score_Final",
        "Melhor_Ponto_Corte"
    ]

    for coluna in colunas_necessarias:
        if coluna not in scores_3x3.columns:
            raise ValueError(
                f"O arquivo {arquivo_scores} precisa ter a coluna '{coluna}'."
            )

    if "Ponto_Corte_Medio" not in scores_3x3.columns:
        scores_3x3["Ponto_Corte_Medio"] = 0.5

    if rank not in scores_3x3["Posicao_Rank"].values:
        raise ValueError(
            f"Rank {rank} não encontrado. "
            f"Ranks disponíveis: 1 até {scores_3x3['Posicao_Rank'].max()}."
        )

    # ========================================================
    # PEGAR COMBINAÇÃO PELO RANK
    # ========================================================

    linha_rank = scores_3x3.loc[
        scores_3x3["Posicao_Rank"] == rank
    ].iloc[0]

    feature_1 = linha_rank["Feature_1"]
    feature_2 = linha_rank["Feature_2"]
    feature_3 = linha_rank["Feature_3"]

    features = [feature_1, feature_2, feature_3]

    melhor_ponto_corte = float(linha_rank["Melhor_Ponto_Corte"])
    ponto_corte_medio = float(linha_rank["Ponto_Corte_Medio"])

    auc_pr = float(linha_rank["AUC_PR"])
    mcc = float(linha_rank["MCC"])
    score_final = float(linha_rank["Score_Final"])

    if "Log_Loss_Norm" in linha_rank.index:
        log_loss_norm = float(linha_rank["Log_Loss_Norm"])

    elif "Log_Loss" in linha_rank.index:
        log_loss_norm = 1 / (1 + float(linha_rank["Log_Loss"]))

    else:
        log_loss_norm = np.nan

    if "Diferenca_Neg_Log_Veross" in linha_rank.index:
        diferenca_neg_log_veross = float(linha_rank["Diferenca_Neg_Log_Veross"])

    elif (
        "Neg_Log_Veross_Com_Rotulo" in linha_rank.index
        and "Neg_Log_Veross_GMM" in linha_rank.index
    ):
        diferenca_neg_log_veross = (
            float(linha_rank["Neg_Log_Veross_Com_Rotulo"])
            - float(linha_rank["Neg_Log_Veross_GMM"])
        )

    else:
        diferenca_neg_log_veross = np.nan

    # ========================================================
    # PREPARAÇÃO DOS DADOS
    # ========================================================

    temp = df[
        [
            feature_1,
            feature_2,
            feature_3,
            target_name
        ]
    ].dropna().copy()

    X = temp[features]
    y_real = temp[target_name].astype(int)

    # ========================================================
    # ESCALONAMENTO E GMM
    # ========================================================

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    gmm = GaussianMixture(
        n_components=2,
        covariance_type="full",
        random_state=42,
        n_init=3,
        reg_covar=1e-6
    )

    gmm.fit(X_scaled)

    clusters = gmm.predict(X_scaled)

    ct = pd.crosstab(clusters, y_real)

    if 1 not in ct.columns:
        raise ValueError("A classe fraude, valor 1, não foi encontrada no target.")

    cluster_fraude = ct[1].idxmax()
    cluster_nao_fraude = 1 - cluster_fraude

    responsabilidades = gmm.predict_proba(X_scaled)

    prob_fraude = responsabilidades[:, cluster_fraude]

    prob_fraude = np.clip(
        prob_fraude,
        1e-15,
        1 - 1e-15
    )

    temp["Responsabilidade_GMM_Fraude"] = prob_fraude

    # ========================================================
    # CORES
    # ========================================================

    colorscale_resp_2d = [
        [0.00, "#3358c9"],
        [0.12, "#5f7fe0"],
        [0.25, "#b9c8f2"],
        [0.40, "#e4e9f8"],
        [0.50, "#f7f7f7"],
        [0.62, "#f6dfd9"],
        [0.75, "#efc1ba"],
        [0.88, "#e7a9ad"],
        [1.00, "#d88997"]
    ]

    cor_nao_fraude = "#2563eb"
    cor_fraude = "#facc15"
    cor_borda = "#111827"

    # ========================================================
    # MATRIZES DE CONFUSÃO
    # ========================================================

    def gerar_matriz_confusao(y_real, probabilidades, threshold):
        y_pred = (probabilidades >= threshold).astype(int)

        cm = confusion_matrix(
            y_real,
            y_pred,
            labels=[0, 1]
        )

        return cm

    def preparar_valores_matriz(cm):

        tn, fp, fn, tp = cm.ravel()

        total_fraudes = fn + tp
        total_nao_fraudes = tn + fp

        fn_pct = fn / total_fraudes * 100 if total_fraudes != 0 else 0
        tp_pct = tp / total_fraudes * 100 if total_fraudes != 0 else 0

        tn_pct = tn / total_nao_fraudes * 100 if total_nao_fraudes != 0 else 0
        fp_pct = fp / total_nao_fraudes * 100 if total_nao_fraudes != 0 else 0

        return {
            "fn": {
                "pct": fn_pct,
                "count": int(fn),
                "qualidade": 100 - fn_pct
            },
            "tp": {
                "pct": tp_pct,
                "count": int(tp),
                "qualidade": tp_pct
            },
            "tn": {
                "pct": tn_pct,
                "count": int(tn),
                "qualidade": tn_pct
            },
            "fp": {
                "pct": fp_pct,
                "count": int(fp),
                "qualidade": 100 - fp_pct
            }
        }

    def preparar_valores_matriz_ideal(y_real):

        y_real_array = np.asarray(y_real).astype(int)

        total_fraudes = int(np.sum(y_real_array == 1))
        total_nao_fraudes = int(np.sum(y_real_array == 0))

        return {
            "fn": {
                "pct": 0.0,
                "count": 0,
                "qualidade": 100.0
            },
            "tp": {
                "pct": 100.0,
                "count": total_fraudes,
                "qualidade": 100.0
            },
            "tn": {
                "pct": 100.0,
                "count": total_nao_fraudes,
                "qualidade": 100.0
            },
            "fp": {
                "pct": 0.0,
                "count": 0,
                "qualidade": 100.0
            }
        }

    def cor_por_qualidade(q):
        if q >= 95:
            return "cell q95"
        elif q >= 85:
            return "cell q85"
        elif q >= 70:
            return "cell q70"
        elif q >= 50:
            return "cell q50"
        elif q >= 30:
            return "cell q30"
        else:
            return "cell q10"

    def gerar_html_matriz(titulo, valores, matriz_ideal=False):

        if matriz_ideal:
            desc_fn = "Erro ideal: nenhuma fraude perdida"
            desc_tp = "Acerto ideal: fraudes detectadas"
            desc_tn = "Acerto ideal: não fraudes corretas"
            desc_fp = "Erro ideal: nenhum falso alerta"
        else:
            desc_fn = "Erro: fraude perdida"
            desc_tp = "Acerto: fraude detectada"
            desc_tn = "Acerto: não fraude"
            desc_fp = "Erro: falso alerta"

        html = f"""
        <section class="matrix-card">
            <h2>{titulo}</h2>

            <div class="matrix-area">

                <div class="matrix-wrapper">

                    <div class="corner"></div>
                    <div class="x-label">Pred Não Fraude</div>
                    <div class="x-label">Pred Fraude</div>

                    <div class="y-label">Real Fraude</div>

                    <div class="{cor_por_qualidade(valores['fn']['qualidade'])}">
                        <div class="pct">{valores['fn']['pct']:.2f}%</div>
                        <div class="count">({formatar_count(valores['fn']['count'])})</div>
                        <div class="cell-desc">{desc_fn}</div>
                    </div>

                    <div class="{cor_por_qualidade(valores['tp']['qualidade'])}">
                        <div class="pct">{valores['tp']['pct']:.2f}%</div>
                        <div class="count">({formatar_count(valores['tp']['count'])})</div>
                        <div class="cell-desc">{desc_tp}</div>
                    </div>

                    <div class="y-label">Real Não Fraude</div>

                    <div class="{cor_por_qualidade(valores['tn']['qualidade'])}">
                        <div class="pct">{valores['tn']['pct']:.2f}%</div>
                        <div class="count">({formatar_count(valores['tn']['count'])})</div>
                        <div class="cell-desc">{desc_tn}</div>
                    </div>

                    <div class="{cor_por_qualidade(valores['fp']['qualidade'])}">
                        <div class="pct">{valores['fp']['pct']:.2f}%</div>
                        <div class="count">({formatar_count(valores['fp']['count'])})</div>
                        <div class="cell-desc">{desc_fp}</div>
                    </div>

                </div>

                <div class="legend">
                    <div class="legend-title">Qualidade</div>
                    <div class="colorbar"></div>
                    <div class="legend-label-top">Melhor</div>
                    <div class="legend-label-bottom">Pior</div>
                </div>

            </div>
        </section>
        """

        return html

    # ========================================================
    # GRÁFICO 3D - CLASSE REAL
    # ========================================================

    def gerar_grafico_3d_classe_real_html(
        df_plot,
        features,
        target_name
    ):
        f1, f2, f3 = features

        dados_nao_fraude = df_plot.loc[
            df_plot[target_name] == 0,
            features
        ].dropna()

        dados_fraude = df_plot.loc[
            df_plot[target_name] == 1,
            features
        ].dropna()

        fig = go.Figure()

        fig.add_trace(
            go.Scatter3d(
                x=dados_nao_fraude[f1],
                y=dados_nao_fraude[f2],
                z=dados_nao_fraude[f3],
                mode="markers",
                name="Não Fraude pontos",
                showlegend=False,
                marker=dict(
                    size=2.7,
                    color=cor_nao_fraude,
                    opacity=0.14
                ),
                hovertemplate=(
                    f"{f1}: %{{x:.4f}}<br>"
                    f"{f2}: %{{y:.4f}}<br>"
                    f"{f3}: %{{z:.4f}}<br>"
                    "Classe: Não Fraude"
                    "<extra></extra>"
                )
            )
        )

        fig.add_trace(
            go.Scatter3d(
                x=dados_fraude[f1],
                y=dados_fraude[f2],
                z=dados_fraude[f3],
                mode="markers",
                name=f"Fraude ({formatar_count(len(dados_fraude))})",
                marker=dict(
                    size=7.5,
                    color=cor_fraude,
                    opacity=0.98,
                    line=dict(
                        color=cor_borda,
                        width=1.2
                    )
                ),
                hovertemplate=(
                    f"{f1}: %{{x:.4f}}<br>"
                    f"{f2}: %{{y:.4f}}<br>"
                    f"{f3}: %{{z:.4f}}<br>"
                    "Classe: Fraude"
                    "<extra></extra>"
                )
            )
        )

        fig.add_trace(
            go.Scatter3d(
                x=[None],
                y=[None],
                z=[None],
                mode="markers",
                name=f"Não Fraude ({formatar_count(len(dados_nao_fraude))})",
                marker=dict(
                    size=10,
                    color=cor_nao_fraude,
                    opacity=1.0,
                    line=dict(
                        color="#1e3a8a",
                        width=1.2
                    )
                ),
                showlegend=True,
                hoverinfo="skip"
            )
        )

        fig.update_layout(
            height=760,
            margin=dict(l=0, r=0, t=35, b=0),
            legend=dict(
                title="Classe Real",
                x=0.70,
                y=0.96,
                bgcolor="rgba(255,255,255,0.92)",
                bordercolor="#d1d5db",
                borderwidth=1,
                font=dict(size=13),
                itemsizing="constant"
            ),
            scene=dict(
                xaxis_title=f1,
                yaxis_title=f2,
                zaxis_title=f3,
                xaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
                yaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
                zaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
                camera=dict(eye=dict(x=1.65, y=1.45, z=0.95))
            )
        )

        return fig_plotly_to_html(fig)

    # ========================================================
    # BOXPLOTS DAS 3 FEATURES
    # ========================================================

    def gerar_boxplots_3_features_html(
        df_plot,
        features,
        target_name
    ):
        fig = make_subplots(
            rows=1,
            cols=3,
            subplot_titles=[
                f"Boxplot - {features[0]}",
                f"Boxplot - {features[1]}",
                f"Boxplot - {features[2]}"
            ],
            horizontal_spacing=0.08
        )

        for idx, feature in enumerate(features, start=1):

            dados_nao_fraude = df_plot.loc[
                df_plot[target_name] == 0,
                feature
            ].dropna()

            dados_fraude = df_plot.loc[
                df_plot[target_name] == 1,
                feature
            ].dropna()

            fig.add_trace(
                go.Box(
                    y=dados_nao_fraude,
                    name="Não Fraude",
                    marker_color=cor_nao_fraude,
                    boxmean=True,
                    opacity=0.75,
                    showlegend=True if idx == 1 else False,
                    hovertemplate=(
                        f"{feature}: %{{y:.4f}}<br>"
                        "Classe: Não Fraude"
                        "<extra></extra>"
                    )
                ),
                row=1,
                col=idx
            )

            fig.add_trace(
                go.Box(
                    y=dados_fraude,
                    name="Fraude",
                    marker_color=cor_fraude,
                    boxmean=True,
                    opacity=0.90,
                    showlegend=True if idx == 1 else False,
                    hovertemplate=(
                        f"{feature}: %{{y:.4f}}<br>"
                        "Classe: Fraude"
                        "<extra></extra>"
                    )
                ),
                row=1,
                col=idx
            )

            fig.update_yaxes(
                title_text=feature,
                row=1,
                col=idx,
                showgrid=True,
                gridcolor="rgba(148,163,184,0.28)",
                zeroline=False
            )

            fig.update_xaxes(
                row=1,
                col=idx,
                showgrid=False
            )

        fig.update_layout(
            height=560,
            margin=dict(l=50, r=40, t=95, b=60),
            legend=dict(
                title="Classe Real",
                orientation="h",
                x=0.5,
                y=1.14,
                xanchor="center",
                yanchor="bottom",
                bgcolor="rgba(255,255,255,0.92)",
                bordercolor="#d1d5db",
                borderwidth=1
            ),
            boxmode="group",
            plot_bgcolor="white",
            paper_bgcolor="white"
        )

        return fig_plotly_to_html(fig)

    # ========================================================
    # GRÁFICO 3D - ELIPSOIDE DO COMPONENTE NÃO FRAUDE
    # ========================================================

    def gerar_grafico_3d_elipsoide_html(
        df_plot,
        features,
        target_name,
        scaler,
        gmm,
        cluster_fraude,
        cluster_nao_fraude,
        threshold,
        raio_elipsoide=3.0
    ):
        f1, f2, f3 = features

        y_real_array = df_plot[target_name].astype(int).to_numpy()
        resp = df_plot["Responsabilidade_GMM_Fraude"].to_numpy()

        mask_nao_fraude = y_real_array == 0
        mask_fraude = y_real_array == 1

        mask_dentro = resp < threshold
        mask_fora = resp >= threshold

        mask_nf_dentro = mask_nao_fraude & mask_dentro
        mask_nf_fora = mask_nao_fraude & mask_fora

        mask_fraude_dentro = mask_fraude & mask_dentro
        mask_fraude_fora = mask_fraude & mask_fora

        # ====================================================
        # MÉDIA E COVARIÂNCIA DO COMPONENTE NÃO FRAUDE
        # NO ESPAÇO ORIGINAL
        # ====================================================

        media_scaled = gmm.means_[cluster_nao_fraude]
        cov_scaled = gmm.covariances_[cluster_nao_fraude]

        scale = scaler.scale_
        mean_scaler = scaler.mean_

        media_original = media_scaled * scale + mean_scaler

        matriz_scale = np.diag(scale)

        cov_original = matriz_scale @ cov_scaled @ matriz_scale

        ell_x, ell_y, ell_z = gerar_elipsoide_3d_media_cov(
            media=media_original,
            cov=cov_original,
            raio=raio_elipsoide,
            n_u=72,
            n_v=36
        )

        fig = go.Figure()

        # ====================================================
        # SUPERFÍCIE DA ELIPSOIDE
        # ====================================================

        fig.add_trace(
            go.Surface(
                x=ell_x,
                y=ell_y,
                z=ell_z,
                opacity=0.22,
                colorscale=[
                    [0.0, "#2563eb"],
                    [1.0, "#2563eb"]
                ],
                showscale=False,
                name="Elipsoide de alta densidade do componente não fraude",
                hovertemplate=(
                    "Elipsoide de alta densidade<br>"
                    "Componente GMM associado à não fraude<br>"
                    f"Raio Mahalanobis = {raio_elipsoide:.2f}<br>"
                    f"Corte = {threshold:.6f}"
                    "<extra></extra>"
                )
            )
        )

        # ====================================================
        # CONTORNOS DA ELIPSOIDE
        # ====================================================

        for k in np.linspace(0, ell_x.shape[1] - 1, 7).astype(int):
            fig.add_trace(
                go.Scatter3d(
                    x=ell_x[:, k],
                    y=ell_y[:, k],
                    z=ell_z[:, k],
                    mode="lines",
                    line=dict(
                        color="#1e3a8a",
                        width=4
                    ),
                    name="Contorno da elipsoide" if k == 0 else None,
                    showlegend=True if k == 0 else False,
                    hoverinfo="skip"
                )
            )

        for k in np.linspace(0, ell_x.shape[0] - 1, 9).astype(int):
            fig.add_trace(
                go.Scatter3d(
                    x=ell_x[k, :],
                    y=ell_y[k, :],
                    z=ell_z[k, :],
                    mode="lines",
                    line=dict(
                        color="#1e3a8a",
                        width=3
                    ),
                    name=None,
                    showlegend=False,
                    hoverinfo="skip"
                )
            )

        # ====================================================
        # PONTOS DENTRO DA REGIÃO ACEITA COMO NÃO FRAUDE
        # ====================================================

        fig.add_trace(
            go.Scatter3d(
                x=df_plot.loc[mask_nf_dentro, f1],
                y=df_plot.loc[mask_nf_dentro, f2],
                z=df_plot.loc[mask_nf_dentro, f3],
                mode="markers",
                name=f"Não fraude dentro da região de aceitação ({formatar_count(mask_nf_dentro.sum())})",
                marker=dict(
                    size=2.6,
                    color="#2563eb",
                    opacity=0.09
                ),
                hovertemplate=(
                    f"{f1}: %{{x:.4f}}<br>"
                    f"{f2}: %{{y:.4f}}<br>"
                    f"{f3}: %{{z:.4f}}<br>"
                    "Classe real: Não Fraude<br>"
                    "GMM: aceito como não fraude<br>"
                    f"Corte = {threshold:.6f}"
                    "<extra></extra>"
                )
            )
        )

        fig.add_trace(
            go.Scatter3d(
                x=df_plot.loc[mask_fraude_dentro, f1],
                y=df_plot.loc[mask_fraude_dentro, f2],
                z=df_plot.loc[mask_fraude_dentro, f3],
                mode="markers",
                name=f"Fraude dentro da região de aceitação ({formatar_count(mask_fraude_dentro.sum())})",
                marker=dict(
                    size=6.3,
                    color="#facc15",
                    opacity=0.62,
                    line=dict(
                        color="#111827",
                        width=0.9
                    )
                ),
                hovertemplate=(
                    f"{f1}: %{{x:.4f}}<br>"
                    f"{f2}: %{{y:.4f}}<br>"
                    f"{f3}: %{{z:.4f}}<br>"
                    "Classe real: Fraude<br>"
                    "GMM: aceito como não fraude<br>"
                    "Interpretação: falso negativo<br>"
                    f"Corte = {threshold:.6f}"
                    "<extra></extra>"
                )
            )
        )

        # ====================================================
        # PONTOS FORA DA REGIÃO ACEITA COMO NÃO FRAUDE
        # ====================================================

        fig.add_trace(
            go.Scatter3d(
                x=df_plot.loc[mask_nf_fora, f1],
                y=df_plot.loc[mask_nf_fora, f2],
                z=df_plot.loc[mask_nf_fora, f3],
                mode="markers",
                name=f"Não fraude fora da região de aceitação ({formatar_count(mask_nf_fora.sum())})",
                marker=dict(
                    size=5.4,
                    color="#dc2626",
                    opacity=0.84,
                    line=dict(
                        color="#7f1d1d",
                        width=0.9
                    )
                ),
                hovertemplate=(
                    f"{f1}: %{{x:.4f}}<br>"
                    f"{f2}: %{{y:.4f}}<br>"
                    f"{f3}: %{{z:.4f}}<br>"
                    "Classe real: Não Fraude<br>"
                    "GMM: classificado como fraude<br>"
                    "Interpretação: falso positivo<br>"
                    f"Corte = {threshold:.6f}"
                    "<extra></extra>"
                )
            )
        )

        fig.add_trace(
            go.Scatter3d(
                x=df_plot.loc[mask_fraude_fora, f1],
                y=df_plot.loc[mask_fraude_fora, f2],
                z=df_plot.loc[mask_fraude_fora, f3],
                mode="markers",
                name=f"Fraude fora da região de aceitação ({formatar_count(mask_fraude_fora.sum())})",
                marker=dict(
                    size=8.8,
                    color=cor_fraude,
                    opacity=0.98,
                    line=dict(
                        color="#111827",
                        width=1.2
                    )
                ),
                hovertemplate=(
                    f"{f1}: %{{x:.4f}}<br>"
                    f"{f2}: %{{y:.4f}}<br>"
                    f"{f3}: %{{z:.4f}}<br>"
                    "Classe real: Fraude<br>"
                    "GMM: classificado como fraude<br>"
                    "Interpretação: fraude detectada<br>"
                    f"Corte = {threshold:.6f}"
                    "<extra></extra>"
                )
            )
        )

        fig.update_layout(
            height=840,
            margin=dict(l=0, r=30, t=40, b=0),
            legend=dict(
                title="Legenda",
                x=0.55,
                y=0.97,
                xanchor="left",
                yanchor="top",
                bgcolor="rgba(255,255,255,0.92)",
                bordercolor="#d1d5db",
                borderwidth=1,
                font=dict(size=13),
                itemsizing="constant"
            ),
            scene=dict(
                xaxis_title=f1,
                yaxis_title=f2,
                zaxis_title=f3,
                xaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
                yaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
                zaxis=dict(backgroundcolor="rgb(248,250,252)", gridcolor="#d1d5db"),
                camera=dict(eye=dict(x=1.65, y=1.45, z=0.95))
            )
        )

        return fig_plotly_to_html(fig)

    # ========================================================
    # SPEARMAN
    # ========================================================

    def gerar_matriz_spearman_3d_html(
        df_plot,
        features,
        target_name
    ):
        dados_corr = df_plot[
            features + [target_name]
        ].copy()

        dados_corr = dados_corr.rename(
            columns={
                target_name: "Fraude"
            }
        )

        corr = dados_corr.corr(method="spearman")

        labels = corr.columns.tolist()

        fig = go.Figure(
            data=go.Heatmap(
                z=corr.values,
                x=labels,
                y=labels,
                zmin=-1,
                zmax=1,
                colorscale="RdBu",
                reversescale=True,
                colorbar=dict(
                    title=dict(
                        text="Spearman",
                        side="top"
                    )
                ),
                text=np.round(corr.values, 3),
                texttemplate="%{text}",
                hovertemplate=(
                    "Linha: %{y}<br>"
                    "Coluna: %{x}<br>"
                    "Spearman: %{z:.6f}"
                    "<extra></extra>"
                )
            )
        )

        fig.update_layout(
            height=620,
            margin=dict(l=80, r=40, t=40, b=80),
            xaxis=dict(tickangle=-35),
            yaxis=dict(autorange="reversed")
        )

        return fig_plotly_to_html(fig)

    # ========================================================
    # PROJEÇÕES 2D INTERATIVAS
    # ========================================================

    def gerar_responsabilidades_3d_projecoes_html(
        df_plot,
        features,
        target_name,
        scaler,
        gmm,
        cluster_fraude,
        threshold
    ):
        f1, f2, f3 = features

        medianas = {
            f1: df_plot[f1].median(),
            f2: df_plot[f2].median(),
            f3: df_plot[f3].median()
        }

        pares = [
            (f1, f2, f3),
            (f1, f3, f2),
            (f2, f3, f1)
        ]

        y_real_array = df_plot[target_name].astype(int).to_numpy()

        mask_nao_fraude = y_real_array == 0
        mask_fraude = y_real_array == 1

        fig = make_subplots(
            rows=1,
            cols=3,
            subplot_titles=[
                f"{f1} x {f2}",
                f"{f1} x {f3}",
                f"{f2} x {f3}",
            ],
            horizontal_spacing=0.07
        )

        for idx, (eixo_x, eixo_y, fixo) in enumerate(pares, start=1):

            x_min = df_plot[eixo_x].min()
            x_max = df_plot[eixo_x].max()

            y_min = df_plot[eixo_y].min()
            y_max = df_plot[eixo_y].max()

            margem_x = 0.05 * (x_max - x_min)
            margem_y = 0.05 * (y_max - y_min)

            x_grid = np.linspace(
                x_min - margem_x,
                x_max + margem_x,
                220
            )

            y_grid = np.linspace(
                y_min - margem_y,
                y_max + margem_y,
                220
            )

            xx, yy = np.meshgrid(x_grid, y_grid)

            grid_df = pd.DataFrame(
                {
                    f1: medianas[f1],
                    f2: medianas[f2],
                    f3: medianas[f3]
                },
                index=np.arange(xx.size)
            )

            grid_df[eixo_x] = xx.ravel()
            grid_df[eixo_y] = yy.ravel()
            grid_df[fixo] = medianas[fixo]

            grid_df = grid_df[features]

            grid_scaled = scaler.transform(grid_df)

            grid_prob = gmm.predict_proba(grid_scaled)[:, cluster_fraude]

            zz = grid_prob.reshape(xx.shape)

            fig.add_trace(
                go.Contour(
                    x=x_grid,
                    y=y_grid,
                    z=zz,
                    colorscale=colorscale_resp_2d,
                    zmin=0,
                    zmax=1,
                    contours=dict(
                        start=0,
                        end=1,
                        size=0.05,
                        coloring="heatmap",
                        showlabels=False
                    ),
                    opacity=0.62,
                    showscale=True if idx == 3 else False,
                    colorbar=dict(
                        title=dict(
                            text="Responsabilidade<br>GMM para fraude",
                            side="right",
                            font=dict(size=13)
                        ),
                        tickvals=[0, 0.25, 0.5, 0.75, 1],
                        ticktext=[
                            "0.00<br>Baixa",
                            "0.25",
                            "0.50",
                            "0.75",
                            "1.00<br>Alta"
                        ],
                        tickfont=dict(size=11),
                        len=0.74,
                        thickness=22,
                        x=1.03,
                        y=0.48,
                        yanchor="middle"
                    ),
                    hovertemplate=(
                        f"{eixo_x}: %{{x:.4f}}<br>"
                        f"{eixo_y}: %{{y:.4f}}<br>"
                        "Resp. GMM: %{z:.6f}"
                        "<extra></extra>"
                    ),
                    name=f"Mapa GMM {eixo_x} x {eixo_y}",
                    showlegend=False
                ),
                row=1,
                col=idx
            )

            fig.add_trace(
                go.Contour(
                    x=x_grid,
                    y=y_grid,
                    z=zz,
                    contours=dict(
                        start=0.25,
                        end=0.75,
                        size=0.25,
                        coloring="none",
                        showlabels=False
                    ),
                    line=dict(
                        color="#6b7280",
                        width=1.1,
                        dash="dot"
                    ),
                    showscale=False,
                    hoverinfo="skip",
                    name="Curvas auxiliares: 0.25, 0.50 e 0.75",
                    showlegend=True if idx == 1 else False
                ),
                row=1,
                col=idx
            )

            fig.add_trace(
                go.Contour(
                    x=x_grid,
                    y=y_grid,
                    z=zz,
                    contours=dict(
                        start=threshold,
                        end=threshold,
                        size=1,
                        coloring="none",
                        showlabels=False
                    ),
                    line=dict(
                        color="#020617",
                        width=3.2,
                        dash="dash"
                    ),
                    showscale=False,
                    hoverinfo="skip",
                    name="Curva de nível da GMM no ponto de corte",
                    showlegend=True if idx == 1 else False
                ),
                row=1,
                col=idx
            )

            fig.add_trace(
                go.Scattergl(
                    x=df_plot.loc[mask_nao_fraude, eixo_x],
                    y=df_plot.loc[mask_nao_fraude, eixo_y],
                    mode="markers",
                    name="Não Fraude pontos",
                    marker=dict(
                        size=3.2,
                        color=cor_nao_fraude,
                        opacity=0.12
                    ),
                    hovertemplate=(
                        f"{eixo_x}: %{{x:.4f}}<br>"
                        f"{eixo_y}: %{{y:.4f}}<br>"
                        "Classe: Não Fraude"
                        "<extra></extra>"
                    ),
                    showlegend=False
                ),
                row=1,
                col=idx
            )

            fig.add_trace(
                go.Scattergl(
                    x=df_plot.loc[mask_fraude, eixo_x],
                    y=df_plot.loc[mask_fraude, eixo_y],
                    mode="markers",
                    name=f"Fraude real ({formatar_count(mask_fraude.sum())})",
                    marker=dict(
                        size=7.5,
                        color=cor_fraude,
                        opacity=0.98,
                        line=dict(
                            color=cor_borda,
                            width=1.1
                        )
                    ),
                    hovertemplate=(
                        f"{eixo_x}: %{{x:.4f}}<br>"
                        f"{eixo_y}: %{{y:.4f}}<br>"
                        "Classe: Fraude"
                        "<extra></extra>"
                    ),
                    showlegend=True if idx == 1 else False
                ),
                row=1,
                col=idx
            )

            if idx == 1:
                fig.add_trace(
                    go.Scattergl(
                        x=[None],
                        y=[None],
                        mode="markers",
                        name=f"Não Fraude real ({formatar_count(mask_nao_fraude.sum())})",
                        marker=dict(
                            size=9,
                            color=cor_nao_fraude,
                            opacity=1.0,
                            line=dict(
                                color="#1e3a8a",
                                width=1.2
                            )
                        ),
                        hoverinfo="skip",
                        showlegend=True
                    ),
                    row=1,
                    col=idx
                )

            fig.update_xaxes(
                title_text=eixo_x,
                row=1,
                col=idx,
                showgrid=True,
                gridcolor="rgba(148,163,184,0.28)",
                zeroline=False
            )

            fig.update_yaxes(
                title_text=eixo_y,
                row=1,
                col=idx,
                showgrid=True,
                gridcolor="rgba(148,163,184,0.28)",
                zeroline=False
            )

        fig.update_layout(
            height=760,
            margin=dict(l=40, r=125, t=135, b=60),
            legend=dict(
                title="Legenda",
                orientation="h",
                x=0.5,
                y=1.10,
                xanchor="center",
                yanchor="bottom",
                bgcolor="rgba(255,255,255,0.92)",
                bordercolor="#d1d5db",
                borderwidth=1,
                font=dict(size=13),
                itemsizing="constant"
            ),
            plot_bgcolor="white",
            paper_bgcolor="white"
        )

        return fig_plotly_to_html(fig)

    # ========================================================
    # GERAR MATRIZES
    # ========================================================

    cm_melhor = gerar_matriz_confusao(
        y_real=y_real,
        probabilidades=prob_fraude,
        threshold=melhor_ponto_corte
    )

    cm_medio = gerar_matriz_confusao(
        y_real=y_real,
        probabilidades=prob_fraude,
        threshold=ponto_corte_medio
    )

    valores_melhor = preparar_valores_matriz(cm_melhor)
    valores_medio = preparar_valores_matriz(cm_medio)
    valores_ideal = preparar_valores_matriz_ideal(y_real)

    # ========================================================
    # GERAR GRÁFICOS
    # ========================================================

    grafico_3d_classe_real_html = gerar_grafico_3d_classe_real_html(
        df_plot=temp,
        features=features,
        target_name=target_name
    )

    boxplots_3_features_html = gerar_boxplots_3_features_html(
        df_plot=temp,
        features=features,
        target_name=target_name
    )

    grafico_3d_elipsoide_melhor_html = gerar_grafico_3d_elipsoide_html(
        df_plot=temp,
        features=features,
        target_name=target_name,
        scaler=scaler,
        gmm=gmm,
        cluster_fraude=cluster_fraude,
        cluster_nao_fraude=cluster_nao_fraude,
        threshold=melhor_ponto_corte,
        raio_elipsoide=raio_elipsoide
    )

    grafico_3d_elipsoide_medio_html = gerar_grafico_3d_elipsoide_html(
        df_plot=temp,
        features=features,
        target_name=target_name,
        scaler=scaler,
        gmm=gmm,
        cluster_fraude=cluster_fraude,
        cluster_nao_fraude=cluster_nao_fraude,
        threshold=ponto_corte_medio,
        raio_elipsoide=raio_elipsoide
    )

    matriz_spearman_html = gerar_matriz_spearman_3d_html(
        df_plot=temp,
        features=features,
        target_name=target_name
    )

    grafico_responsabilidades_melhor_html = gerar_responsabilidades_3d_projecoes_html(
        df_plot=temp,
        features=features,
        target_name=target_name,
        scaler=scaler,
        gmm=gmm,
        cluster_fraude=cluster_fraude,
        threshold=melhor_ponto_corte
    )

    grafico_responsabilidades_medio_html = gerar_responsabilidades_3d_projecoes_html(
        df_plot=temp,
        features=features,
        target_name=target_name,
        scaler=scaler,
        gmm=gmm,
        cluster_fraude=cluster_fraude,
        threshold=ponto_corte_medio
    )

    # ========================================================
    # HTML DAS MATRIZES
    # ========================================================

    combinacao_nome = f"{feature_1} + {feature_2} + {feature_3}"

    html_melhor = gerar_html_matriz(
        titulo=f"Matriz de Confusão (%) - {combinacao_nome} - Melhor Ponto de Corte",
        valores=valores_melhor
    )

    html_medio = gerar_html_matriz(
        titulo=f"Matriz de Confusão (%) - {combinacao_nome} - Ponto de Corte 0.5",
        valores=valores_medio
    )

    html_ideal = gerar_html_matriz(
        titulo="Matriz de Confusão Ideal (%)",
        valores=valores_ideal,
        matriz_ideal=True
    )

    # ========================================================
    # HTML FINAL
    # ========================================================

    html_final = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <title>Relatório 3x3 Interativo - Rank {rank} - {feature_1} + {feature_2} + {feature_3}</title>

        <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

        <style>
            body {{
                font-family: Arial, Helvetica, sans-serif;
                background: #f4f6f8;
                color: #020617;
                margin: 0;
                padding: 32px;
            }}

            .container {{
                max-width: 1500px;
                margin: 0 auto;
            }}

            h1 {{
                text-align: center;
                margin-bottom: 28px;
                color: #020617;
            }}

            .info-box {{
                background: #ffffff;
                border-radius: 16px;
                padding: 20px 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .info-grid {{
                display: grid;
                grid-template-columns: repeat(3, 1fr);
                gap: 14px;
                margin-top: 14px;
            }}

            .info-item {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                padding: 12px 14px;
            }}

            .info-label {{
                font-size: 13px;
                font-weight: 700;
                color: #475569;
                margin-bottom: 6px;
            }}

            .info-value {{
                font-size: 18px;
                font-weight: 800;
                color: #020617;
                font-family: Consolas, Monaco, monospace;
            }}

            .matrix-card {{
                background: white;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .matrix-card h2,
            .plot-card h2 {{
                text-align: center;
                margin-top: 0;
                margin-bottom: 24px;
                color: #020617;
                font-size: 22px;
            }}

            .matrix-area {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 34px;
            }}

            .matrix-wrapper {{
                display: grid;
                grid-template-columns: 180px 1fr 1fr;
                grid-template-rows: 48px 190px 190px;
                width: 950px;
            }}

            .corner {{
                background: transparent;
            }}

            .x-label {{
                display: flex;
                align-items: center;
                justify-content: center;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
                border-bottom: 1px solid #e5e7eb;
            }}

            .y-label {{
                display: flex;
                align-items: center;
                justify-content: flex-end;
                padding-right: 18px;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
            }}

            .cell {{
                display: flex;
                flex-direction: column;
                align-items: center;
                justify-content: center;
                min-height: 180px;
                border: 1px solid #e5e7eb;
                font-size: 20px;
                text-align: center;
                color: #020617 !important;
            }}

            .pct {{
                font-size: 30px;
                font-weight: 900;
                margin-bottom: 4px;
                color: #020617 !important;
            }}

            .count {{
                font-size: 24px;
                font-weight: 900;
                margin-bottom: 8px;
                color: #020617 !important;
            }}

            .cell-desc {{
                font-size: 13px;
                font-weight: 700;
                opacity: 1;
                color: #020617 !important;
            }}

            .q95 {{ background: #08306b; }}
            .q85 {{ background: #08519c; }}
            .q70 {{ background: #2171b5; }}
            .q50 {{ background: #6baed6; }}
            .q30 {{ background: #c6dbef; }}
            .q10 {{ background: #eff6ff; }}

            .legend {{
                position: relative;
                display: flex;
                flex-direction: column;
                align-items: center;
                min-width: 115px;
            }}

            .legend-title {{
                font-weight: 800;
                font-size: 15px;
                margin-bottom: 10px;
                color: #020617;
            }}

            .colorbar {{
                width: 30px;
                height: 310px;
                border-radius: 16px;
                background: linear-gradient(
                    to bottom,
                    #08306b 0%,
                    #08519c 18%,
                    #2171b5 36%,
                    #6baed6 58%,
                    #c6dbef 78%,
                    #eff6ff 100%
                );
                border: 1px solid #cbd5e1;
            }}

            .legend-label-top {{
                position: absolute;
                top: 43px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .legend-label-bottom {{
                position: absolute;
                top: 335px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .plot-card {{
                background: white;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
                overflow-x: auto;
            }}

            .plotly-graph-div {{
                width: 100% !important;
            }}
        </style>
    </head>

    <body>
        <div class="container">

            <h1>Relatório 3x3 Interativo - Rank {rank} - {feature_1} + {feature_2} + {feature_3}</h1>

            <div class="info-box">
                <div class="info-grid">

                    <div class="info-item">
                        <div class="info-label">Rank</div>
                        <div class="info-value">{rank}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Feature 1</div>
                        <div class="info-value">{feature_1}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Feature 2</div>
                        <div class="info-value">{feature_2}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Feature 3</div>
                        <div class="info-value">{feature_3}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Melhor Ponto de Corte</div>
                        <div class="info-value">{melhor_ponto_corte:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Ponto de Corte Médio</div>
                        <div class="info-value">{ponto_corte_medio:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">AUC-PR</div>
                        <div class="info-value">{auc_pr:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">MCC</div>
                        <div class="info-value">{mcc:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Log Loss Norm</div>
                        <div class="info-value">{log_loss_norm:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Score Final</div>
                        <div class="info-value">{score_final:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Diferença Neg. Log-Veross.</div>
                        <div class="info-value">{diferenca_neg_log_veross:.6f}</div>
                    </div>

                </div>
            </div>

            {html_melhor}

            {html_medio}

            {html_ideal}

            <section class="plot-card">
                <h2>Dispersão 3D Interativa por Classe Real - {feature_1} x {feature_2} x {feature_3}</h2>
                {grafico_3d_classe_real_html}
            </section>

            <section class="plot-card">
                <h2>Boxplots das Features por Classe - {feature_1}, {feature_2} e {feature_3}</h2>
                {boxplots_3_features_html}
            </section>

            <section class="plot-card">
                <h2>Elipsoide de Alta Densidade do Componente GMM de Não Fraude com corte {melhor_ponto_corte:.6f} - {feature_1} x {feature_2} x {feature_3}</h2>
                {grafico_3d_elipsoide_melhor_html}
            </section>

            <section class="plot-card">
                <h2>Elipsoide de Alta Densidade do Componente GMM de Não Fraude com corte {ponto_corte_medio:.6f} - {feature_1} x {feature_2} x {feature_3}</h2>
                {grafico_3d_elipsoide_medio_html}
            </section>

            <section class="plot-card">
                <h2>Correlação de Spearman entre Features e Target - {feature_1}, {feature_2}, {feature_3} e Fraude</h2>
                {matriz_spearman_html}
            </section>

            <section class="plot-card">
                <h2>Responsabilidades Estimadas pela GMM - Melhor Ponto de Corte - Projeções 2D</h2>
                {grafico_responsabilidades_melhor_html}
            </section>

            <section class="plot-card">
                <h2>Responsabilidades Estimadas pela GMM - Ponto de Corte Médio - Projeções 2D</h2>
                {grafico_responsabilidades_medio_html}
            </section>

        </div>
    </body>
    </html>
    """

    # ========================================================
    # SALVAR HTML
    # ========================================================

    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    nome_arquivo = (
        f"{nome_base_saida}_rank_{rank}_"
        f"{sanitizar_nome(feature_1)}__"
        f"{sanitizar_nome(feature_2)}__"
        f"{sanitizar_nome(feature_3)}.html"
    )

    caminho_html = pasta_saida / nome_arquivo

    caminho_html.write_text(
        html_final,
        encoding="utf-8"
    )

    print(f"HTML interativo gerado com sucesso: {caminho_html.resolve()}")

    return caminho_html

In [20]:
gerar_relatorio_3x3_interativo(rank=1)
gerar_relatorio_3x3_interativo(rank=2)
gerar_relatorio_3x3_interativo(rank=3)

HTML interativo gerado com sucesso: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\3x3_relat_interativo_rank_1_V11__V17__V26.html
HTML interativo gerado com sucesso: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\3x3_relat_interativo_rank_2_V14__V17__V26.html
HTML interativo gerado com sucesso: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\3x3_relat_interativo_rank_3_V14__V17__tempo_desde_a_primeira_transacao.html


WindowsPath('3x3_relat_interativo_rank_3_V14__V17__tempo_desde_a_primeira_transacao.html')